# Models

Looking at the lower level API of Transformers - the models that wrap PyTorch code for the transformers themselves.

This notebook can run on a low-cost.


In [1]:
%pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

/Users/pradeepkumar/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential = True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

it's faster model :
https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

In [4]:
LLAMA = "meta-llama/Meta-Llama-3.1-8B"
LLAMA_INSTRUCT = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [5]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

In [6]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [7]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA_INSTRUCT)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("mps")

---

## Model Tokenization and Preparation

### 1. Loading the Tool
`tokenizer = AutoTokenizer.from_pretrained(LLAMA_INSTRUCT)`

* **What it does:** This pulls the specific "vocabulary" and "grammar rules" used by the Llama model you’ve selected.
* **Why it matters:** Every model has its own unique way of slicing words (tokenization). For example, one model might see "strawberry" as one token, while another sees "straw" + "berry." You must use the exact tokenizer the model was trained with, or the resulting numbers will be gibberish to the model.

### 2. Handling the "Padding" Problem
`tokenizer.pad_token = tokenizer.eos_token`

* **The Problem:** Neural networks process data in batches and require all sequences in a batch to be the same length. If Sentence A is 5 tokens and Sentence B is 10, Sentence A needs "filler" (padding).
* **The Fix:** Many modern LLMs don't have a dedicated "padding token." This line tells the tokenizer: *"Whenever you need to fill empty space to equalize lengths, just use the End-of-Sentence (EOS) token."*

### 3. The "Chat Template" Transformation
`inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("mps")`

This single line performs three critical heavy-lifting tasks:

1.  **`apply_chat_template(messages)`**: Instruct-tuned models require specific formatting tags to distinguish between the user and the AI (e.g., `<|user|>` or `[INST]`). This wraps your raw text in the exact syntax the model was trained to recognize.
2.  **`return_tensors="pt"`**: This converts the list of tokens into a **PyTorch Tensor**, the mathematical format required by the model's neural network.
3.  **`.to("mps")`**: This moves the data to your hardware. 
    * **`mps`**: Metal Performance Shaders (Apple Silicon M1/M2/M3 chips).
    * **`cuda`**: Use this instead if you are on an NVIDIA GPU.
    * **`cpu`**: Use this if no GPU is available.

---

In [8]:
inputs

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
            315,   2956,  57116, 128009]], device='mps:0')

In [ ]:
# The model

model = AutoModelForCausalLM.from_pretrained(LLAMA_INSTRUCT, device_map="auto", quantization_config=quant_config)

# Breakdown: AutoModelForCausalLM.from_pretrained

This line of code is the "bread and butter" of loading modern Large Language Models (LLMs) using the Hugging Face transformers library. It essentially tells your computer: "Go find this specific brain, shrink it so it fits in my memory, and figure out the best way to run it on my hardware."

---

### 1. `AutoModelForCausalLM`
This is a **loader class**. Instead of manually importing a specific class (like `LlamaForCausalLM`), the "Auto" class reads the model's configuration file and automatically selects the correct architecture.
* **CausalLM:** Refers to "decoder-only" models (Llama, GPT, Mistral) designed to predict the next token in a sequence.

### 2. `.from_pretrained(LLAMA)`
* **`LLAMA`:** This variable holds the Model ID (e.g., `"meta-llama/Llama-2-7b-hf"`) or a local path to the model weights.
* **Function:** It handles downloading the weights and loading them into tensors.

### 3. `device_map="auto"`
This utilizes the `accelerate` library to handle **Model Parallelism**.
* **Smart Allocation:** It calculates available VRAM on your GPU(s) and RAM on your CPU. 
* **Prioritization:** It loads as many layers as possible onto the GPU for speed, then "spills over" any remaining layers to the CPU.



### 4. `quantization_config=quant_config`
Large models are usually 16-bit or 32-bit, which requires massive memory. 
* **Quantization:** Compresses weights (usually to 4-bit or 8-bit) to save memory with minimal loss in accuracy.
* **`quant_config`:** A configuration object (using `BitsAndBytesConfig`) that defines the specific compression method used.

---

### Summary Table: Why use this?

| Feature | Without this code | With this code |
| :--- | :--- | :--- |
| **Memory Usage** | Might require 24GB+ VRAM for a 7B model. | Can fit a 7B model into ~5-6GB VRAM. |
| **Hardware** | Requires a high-end enterprise GPU. | Can run on consumer-grade gaming laptops. |
| **Setup** | Manual layer placement and manual loading. | Fully automated hardware detection. |

> **Note:** To run this, you must have `transformers`, `accelerate`, and `bitsandbytes` installed.

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

---

## Checking Resource Usage

`memory = model.get_memory_footprint() / 1e6`
`print(f"Memory footprint: {memory:,.1f} MB")`

### What this does:
* **`get_memory_footprint()`**: This method calculates the total memory occupied by the model's weights (parameters) in bytes.
* **`/ 1e6`**: Converts the value from bytes to **Megabytes (MB)** to make it human-readable.
* **`f"{memory:,.1f}"`**: Formats the output to include a comma for thousands and rounds it to one decimal place.

### Why it is useful:
When working with quantized models (4-bit or 8-bit), this allows you to verify exactly how much VRAM you've saved. It helps you monitor if the model fits comfortably within your hardware limits or if you are at risk of an **Out of Memory (OOM)** error.



---

## Looking under the hood at the Transformer model

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: 
(a) self-attention layers 
(b) multi-layer perceptron (MLP) layers 
(c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [ ]:
# Execute this cell and look at what gets printed; investigate the layers

model

### And if we want to go even deeper into Transformers

In addition to looking at each of the layers in the model, we can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:  
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

In [ ]:
# let's run the model!

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

In [ ]:
tokenizer.decode(outputs[0])

In [ ]:
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## A couple of quick notes on the next block of code:

I'm using a HuggingFace utility called TextStreamer so that results stream back.
To stream results, we simply replace:  
`outputs = model.generate(inputs, max_new_tokens=80)`  
With:  
`streamer = TextStreamer(tokenizer)`  
`outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)`

Also I've added the argument `add_generation_prompt=True` to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

Thank you to student Piotr B for raising the issue!

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)



In [ ]:
generate(PHI, messages)